# Encoding, Scaling, Feature Selection

# Import Libraries

In [6]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from math import ceil
import math


# Create Meta Data

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



# Import Dataset

In [7]:
sample = pd.read_csv('../data/sample_submission.csv')
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [8]:
sample.set_index('carID', inplace = True)
train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

## Define the independent variables as X and the dependent as Y

In [9]:
X = train.drop('price', axis = 1)
y = train['price']

## Feature Engineering

On this step of the project we create new variables that make sense and can help us predicting our target. For now the variables that we want to add are:
 - `car_age:` calculate the age of the car considering the current year and the year of his registration. This feature help us to capture depreciation, this is, older cars tend to be cheaper.
- `is_recent_model:` binary flag: 1 if the car is less than 3 years old, otherwise 0. We decided to create this feature because highlights newer cars, usually have higher resale prices.

- `mileage_per_year:` used to create the next new feature
- `usage_category:` Discretized version of mileage_per_year (Low / Medium / High).

- `is_hybrid_or_eletric:` binary indicator that is 1 if the fuel type is Hybrid or Electric. Eco-friendly cars often hold higher market value.
- `is_automatic:`binary indicator that is 1 if the type of transmission is automatic or semi-automatic. Automatics typically cost more to buy and maintain.

- `fuel_eficciency_score:`

- `tax_to_engine_ratio:`

- `tax_efficency:`

- `paint_quality_category:` Paint quality binned into Low, Medium, High.

- `has_damage_or_low_paint:`

- `is_first_owner:` binary indicator that is 1 if there were no previous owners. First-owner cars tend to have less wear, better care and higher prices.

- `value_per_engine:`

- `value_per_mileage:`

To do this we'll implement some functions that help us defining this new variables.

In [ ]:
def calculate_car_age(X, current_year=2020): 
    """Calculate car age from the registration year."""
    X["car_age"] = current_year - X["year"]
    return X

def create_mileage_category(X):
    """Creates a simple categorical feature 'mileage_category' based on the total mileage of each car."""
    bins = [0, 10_000, 50_000, 100_000, 150_000, float('inf')]
    labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

    X['mileage_category'] = pd.cut(X['mileage'], bins=bins, labels=labels, include_lowest=True)
    return X

def fuel_transmission_features(X):
    """Binary flags for eco-friendly and automatic cars."""
    X["is_hybrid_or_electric"] = X["fuelType"].isin(["Hybrid", "Electric"]).astype(int)
    X["is_automatic"] = X["transmission"].isin(["Automatic", "Semi-Auto"]).astype(int)
    X["fuel_efficiency_score"] = X["mpg"] / X["engineSize"]
    return X


def tax_and_efficiency_features(X):
    """Economic indicators based on tax and engine size."""
    X["tax_to_engine_ratio"] = X["tax"] / X["engineSize"]
    X["tax_efficiency"] = X["mpg"] / X["tax"]
    return X

def condition_features(X):
    """Simplify paint quality and damage info."""
    X["paintQuality_category"] = pd.cut(X["paintQuality%"],
                                         bins=[0, 40, 70, 100],
                                         labels=["Low", "Medium", "High"])
    X["has_damage_or_low_paint"] = ((X["hasDamage"] == 1) | (X["paintQuality%"] < 40)).astype(int)
    return X

def ownership_features(X):
    """Ownership-related flags."""
    X["is_first_owner"] = (X["previousOwners"] == 0).astype(int)
    X["ownership_ratio"] = np.where(X["car_age"] > 0,
                                     X["previousOwners"] / X["car_age"],
                                     0)
    return X


# Apply all feature functions
X_fe = X.copy()

X_fe = (X_fe
        .pipe(calculate_car_age)
        .pipe(create_mileage_category)
        .pipe(fuel_transmission_features)
        .pipe(tax_and_efficiency_features)
        .pipe(condition_features)
        .pipe(ownership_features)
       )

# Inspect results
print("New columns created:")
print([col for col in X_fe.columns if col not in X.columns])



print('--------------------------------------------------------------------')
print('Now we visualize the dataframe with the new features created to understand them better:')

novas_features = [col for col in X_fe.columns if col not in X.columns]
X_fe[novas_features].tail(20)

New columns created:
['car_age', 'mileage_category', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'paintQuality_category', 'has_damage_or_low_paint', 'is_first_owner', 'ownership_ratio']
--------------------------------------------------------------------
Now we visualize the dataframe with the new features created to understand them better:


,car_age,mileage_category,is_hybrid_or_electric,is_automatic,fuel_efficiency_score,tax_to_engine_ratio,tax_efficiency,paintQuality_category,has_damage_or_low_paint,is_first_owner,ownership_ratio
carID,,,,,,,,,,,
71932,5.0,Very Low,0,1,28.533333,96.666667,0.295172,Low,1,0,0.800000
28693,8.0,Low,0,0,30.050000,72.500000,0.414483,Medium,0,0,0.500000
53707,8.0,Very Low,0,0,42.307692,23.076923,1.833333,Medium,0,0,0.125000
5311,6.0,Very Low,0,1,19.100000,75.000000,0.254667,Low,1,0,0.333333
67969,6.0,Very Low,0,1,16.250000,72.500000,0.224138,Medium,0,0,0.500000
64925,8.0,Low,0,0,61.400000,20.000000,3.070000,High,0,0,0.250000
62955,8.0,Low,0,0,39.571429,107.142857,0.369333,High,0,0,0.500000
59735,7.0,Low,0,1,45.250000,120.833333,0.374483,Medium,0,0,0.285714
769,9.0,Low,0,0,67.300000,0.000000,inf,Low,1,0,0.444444
